## All imports

In [31]:
import pandas as pd
import numpy as np
import torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import logging
from tqdm import tqdm
import nltk
nltk.download('punkt')
nltk.download('wordnet')
from nltk.stem.porter import PorterStemmer
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords


[nltk_data] Downloading package punkt to /home/nicolas/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /home/nicolas/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


## Data handling, understanding etc

In [11]:
# Reading Data
df=pd.read_csv("fake_news_dataset.csv")

In [12]:
# Description of the classes in our dataset
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20800 entries, 0 to 20799
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   id      20800 non-null  int64
 1   title   20242 non-null  str  
 2   author  18843 non-null  str  
 3   text    20761 non-null  str  
 4   label   20800 non-null  int64
dtypes: int64(2), str(3)
memory usage: 94.6 MB


In [13]:
# Exemples 
df.head(10)

,id,title,author,text,label
0,0,House Dem Aide: We Didn’t Even See Comey’s Let...,Darrell Lucus,House Dem Aide: We Didn’t Even See Comey’s Let...,1
1,1,"FLYNN: Hillary Clinton, Big Woman on Campus - ...",Daniel J. Flynn,Ever get the feeling your life circles the rou...,0
2,2,Why the Truth Might Get You Fired,Consortiumnews.com,"Why the Truth Might Get You Fired October 29, ...",1
3,3,15 Civilians Killed In Single US Airstrike Hav...,Jessica Purkiss,Videos 15 Civilians Killed In Single US Airstr...,1
4,4,Iranian woman jailed for fictional unpublished...,Howard Portnoy,Print \nAn Iranian woman has been sentenced to...,1
5,5,Jackie Mason: Hollywood Would Love Trump if He...,Daniel Nussbaum,"In these trying times, Jackie Mason is the Voi...",0
6,6,Life: Life Of Luxury: Elton John’s 6 Favorite ...,NaN,Ever wonder how Britain’s most iconic pop pian...,1
7,7,Benoît Hamon Wins French Socialist Party’s Pre...,Alissa J. Rubin,"PARIS — France chose an idealistic, traditi...",0
8,8,Excerpts From a Draft Script for Donald Trump’...,NaN,Donald J. Trump is scheduled to make a highly ...,0
9,9,"A Back-Channel Plan for Ukraine and Russia, Co...",Megan Twohey and Scott Shane,A week before Michael T. Flynn resigned as nat...,0


## Pre-Processing functions (utils)

In [ ]:
# Multiple functions to pre-process the data

# All text from "title" and "text" is put in lowercase
def to_lowercase(df):
    if "combined_text" in df.columns:
        df["combined_text"] = df["combined_text"].str.lower()
    df["title"] = df["title"].str.lower()
    df["text"] = df["text"].str.lower()
    return df

# Remove punctuation from the text
def delete_punctuation(df):
    df['text'] = df['text'].str.replace(r'[.,!?:";()—–]', '', regex=True)
    if "combined_text" in df.columns:
        df["combined_text"] = df["combined_text"].str.replace(r'[.,!?":;()—–]', '', regex=True)
    return df

# Stemming 
def stem_text(df):
    stemmer = PorterStemmer()
    target_col = "combined_text" if "combined_text" in df.columns else "text"

    df[target_col] = (
        df[target_col].fillna("").astype(str).apply(lambda x: " ".join(stemmer.stem(word) for word in x.split()))
    )

    return df

# Lemmatisation
def lemmatize_text(df):
    lemmatizer=WordNetLemmatizer()
    target_col = "combined_text" if "combined_text" in df.columns else "text"

    df[target_col] = (
        df[target_col].fillna("").astype(str).apply(lambda x: " ".join(lemmatizer.lemmatize(word) for word in x.split()))
    )
    return df

# Combining the "title" and "text" columns into a single column for better processing into a new column "combined_text"
def combine_text_title(df):
    df["combined_text"] = df["title"] + " " + df["text"]
    return df

stop_words = set(stopwords.words("english"))

# Remove stopwords
def remove_stopwords(df):
    df["combined_text"] = df["combined_text"].apply(
        lambda x: " ".join(word for word in x.split() if word not in stop_words)
    )
    return df

## All tests

In [ ]:
dftest = df.copy()
dftest = combine_text_title(dftest)
dftest.head(10)
dftest = to_lowercase(dftest)
dftest.head(10)
dftest = delete_punctuation(dftest)
dftest.head(10)
# dftest = stem_text(dftest)
# dftest.head(10)
dftest = lemmatize_text(dftest)
dftest.head(10)


,id,title,author,text,label,combined_text
0,0,house dem aide: we didn’t even see comey’s let...,Darrell Lucus,house dem aide we didn’t even see comey’s lett...,1,house dem aide we didn’t even see comey’s lett...
1,1,"flynn: hillary clinton, big woman on campus - ...",Daniel J. Flynn,ever get the feeling your life circles the rou...,0,flynn hillary clinton big woman on campus - br...
2,2,why the truth might get you fired,Consortiumnews.com,why the truth might get you fired october 29 2...,1,why the truth might get you fired why the trut...
3,3,15 civilians killed in single us airstrike hav...,Jessica Purkiss,videos 15 civilians killed in single us airstr...,1,15 civilian killed in single u airstrike have ...
4,4,iranian woman jailed for fictional unpublished...,Howard Portnoy,print \nan iranian woman has been sentenced to...,1,iranian woman jailed for fictional unpublished...
5,5,jackie mason: hollywood would love trump if he...,Daniel Nussbaum,in these trying times jackie mason is the voic...,0,jackie mason hollywood would love trump if he ...
6,6,life: life of luxury: elton john’s 6 favorite ...,NaN,ever wonder how britain’s most iconic pop pian...,1,life life of luxury elton john’s 6 favorite sh...
7,7,benoît hamon wins french socialist party’s pre...,Alissa J. Rubin,paris france chose an idealistic tradition...,0,benoît hamon win french socialist party’s pres...
8,8,excerpts from a draft script for donald trump’...,NaN,donald j trump is scheduled to make a highly a...,0,excerpt from a draft script for donald trump’s...
9,9,"a back-channel plan for ukraine and russia, co...",Megan Twohey and Scott Shane,a week before michael t flynn resigned as nati...,0,a back-channel plan for ukraine and russia cou...


## Preprocess data

In [ ]:
def preprocess_ml(df, use_stemming=False):
    df=df.dropna(subset=["text"])
    df = combine_text_title(df)
    df = to_lowercase(df)
    df = delete_punctuation(df)
    df = remove_stopwords(df)
    
    if use_stemming:
        df = stem_text(df)
    else:
        df = lemmatize_text(df)
    
    return df

def preprocess_transformer(df):
    df=df.dropna(subset=["text"])
    df = combine_text_title(df)
    df = to_lowercase(df)
    df = delete_punctuation(df)
    return df

df_processed_ml = preprocess_ml(df)
df_processed_transformer = preprocess_transformer(df)



## Visualization

# Models 

### Logistic Regression

### SVM

### Transformers